In [ ]:
#@title pip installs
# Download SDK for google ai
!pip install -q -U google-generativeai

# install dependecies for image handling
!pip install -q objaverse trimesh
!pip install -q accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 737.0/737.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.3 MB/s eta 0:00:00


In [ ]:
#@title imports
import google.generativeai as genai
from google.colab import userdata
from PIL import Image
import os
import time
import sys
import torch
import objaverse
import trimesh
import numpy as np
import cv2
import glob
import shutil
from diffusers import StableDiffusionPipeline

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
#@title mount drive
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/ Generative Models Course"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#@title get pytorch3D handler file
sys.path.append(drive_path)
try:
  import pytorch3D_handler as pt3dh
  print(" 'pytorch3D_handler.py' imported successfully as pt3dh.")
except Exception as e:
  print(f"Error importing 'functions.py': {e}")

In [ ]:
#@title reload pytorch3D handler
import importlib
importlib.reload(pt3dh)
print("✅ pytorch3D_handler.py reloaded successfully!")

In [ ]:
#@title manage pytorch3D installation - DON'T RUN THIS!

# uncomment and run:
#pt3dh.maual_install_pytorch3D()

🖥️ Detected GPU: NVIDIA L4
✅ Found cached wheel in /content/drive/MyDrive/wheels_pytorch3D_NVIDIA_L4. Installing...
Processing ./drive/MyDrive/wheels_pytorch3D_NVIDIA_L4/iopath-0.1.10-py3-none-any.whl
iopath is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
🎉 PyTorch3D is ready!


In [ ]:
#@title Run this to install pytorch3D!

# 2. Define Path
gpu_name = torch.cuda.get_device_name(0)
if "T4" in gpu_name:
    wheel_folder = "/content/drive/MyDrive/wheels_pytorch3D_T4"
elif "L4" in gpu_name or "A100" in gpu_name:
    wheel_folder = f"/content/drive/MyDrive/wheels_pytorch3D_{gpu_name.replace(' ', '_')}"
else:
    wheel_folder = "/content/drive/MyDrive/wheels_pytorch3D_Generic"

print(f"📂 Looking for PyTorch3D wheel in: {wheel_folder}")

# 3. Find and Install the CORRECT Wheel
if os.path.exists(wheel_folder):
    # Filter list to find ONLY the file containing "pytorch3d"
    files = [f for f in os.listdir(wheel_folder) if f.endswith('.whl') and "pytorch3d" in f]

    if len(files) > 0:
        # Pick the most recent one if multiple exist
        files.sort()
        wheel_path = os.path.join(wheel_folder, files[-1])
        print(f"   🎯 Found PyTorch3D wheel: {files[-1]}")
        print("   ⏳ Installing... (This takes ~30 seconds)")

        !pip install "{wheel_path}"

        print("✅ Installation complete.")
    else:
        print(f"❌ No 'pytorch3d' wheel found in {wheel_folder}.")
        print("   (Found only: " + str([f for f in os.listdir(wheel_folder) if f.endswith('.whl')]) + ")")
        print("   -> You may need to run the build script again.")
else:
    print(f"❌ Folder {wheel_folder} not found.")

# 4. Verify
print("-" * 30)
try:
    import pytorch3d
    from pytorch3d.io import load_objs_as_meshes
    print(f"✅ SUCCESS! PyTorch3D {pytorch3d.__version__} is active.")
except ImportError as e:
    print(f"❌ FAILURE: Still cannot import PyTorch3D.\nError: {e}")

📂 Looking for PyTorch3D wheel in: /content/drive/MyDrive/wheels_pytorch3D_NVIDIA_L4
   🎯 Found PyTorch3D wheel: pytorch3d-0.7.9-cp312-cp312-linux_x86_64.whl
   ⏳ Installing... (This takes ~30 seconds)
Processing ./drive/MyDrive/wheels_pytorch3D_NVIDIA_L4/pytorch3d-0.7.9-cp312-cp312-linux_x86_64.whl
pytorch3d is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
✅ Installation complete.
------------------------------
✅ SUCCESS! PyTorch3D 0.7.9 is active.


In [ ]:
#@title get functions file
sys.path.append(drive_path)
try:
  import functions as fn
  print(" 'functions.py' imported successfully as fn.")
except Exception as e:
  print(f"Error importing 'functions.py': {e}")

 'functions.py' imported successfully as fn.


In [ ]:
#@title reload functions
import importlib
importlib.reload(fn)
print("✅ functions.py reloaded successfully!")

✅ functions.py reloaded successfully!


In [ ]:
#@title configure gemini key

# 1. Retrieve Key
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    print(f"✅ Key found! Starts with: {api_key[:4]}...")
except Exception as e:
    print(f"❌ Error: Could not find 'GOOGLE_API_KEY'. Check the Secrets tab!\nDetails: {e}")

# 2. Configure Gemini
if 'api_key' in locals():
    genai.configure(api_key=api_key)
    print("✅ Gemini Configured successfully.")

✅ Key found! Starts with: AIza...
✅ Gemini Configured successfully.


In [ ]:
#@title define path to images

image_path = f"{drive_path}/combined_images"

if os.path.exists(image_path):
  files = os.listdir(image_path)
  print(f"Found {len(files)} files: {files[:5]}") # prints first 5 files
else:
  print(f"Image path not found: {image_path}")

Found 7 files: ['4a40780198964bdeb6e9b6950a3c69fb_combined.jpg', 'aec69c979166446eb2c8e1503f570d26_combined.jpg', 'bdc8eb3e2a10409bafdfa85fca906204_combined.jpg', '439c61b8881b483688be86e7ad0cbd1e_combined.jpg', 'c94d08a3c24349bbbbd7213532cd05ee_combined.jpg']


Now that we have the prompt generator we need to get the images. The following code attempts to do just that.

Up to here is data loader functions and operations.
From here is the Model load and training, using the data and function we prepared in the first part.

In [ ]:
#@title Model Management System

# Load the base model
vae, tokenizer, text_encoder, unet, noise_scheduler = fn.load_stable_diffusion_model()

# Setup Optimizer (Standard for SD)
optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-5)

# Check for Drive Checkpoints
# This will start at Epoch 0 now, but if you run it tomorrow it will resume!
checkpoint_folder = f"{drive_path}/checkpoints"
start_epoch, _ = fn.load_sd_checkpoint(unet, optimizer, checkpoint_folder)

print(f"Ready to train from Epoch {start_epoch}!")

⏳ Loading SD v1.5 from runwayml/stable-diffusion-v1-5...
🔄 Found checkpoint at /content/drive/MyDrive/ Generative Models Course/checkpoints/sd_finetune.pth. Loading...
Ready to train from Epoch 101!


In [ ]:
# --- DEEP PROBE DIAGNOSTIC ---
import trimesh
import torch
import numpy as np
import os
from pytorch3d.io import load_objs_as_meshes
from pytorch3d.renderer import (
    FoVPerspectiveCameras, look_at_view_transform,
    RasterizationSettings, MeshRenderer, MeshRasterizer,
    SoftPhongShader, PointLights
)

# 1. Setup
print("🔍 Starting Deep Probe Diagnostic...")
device = torch.device("cuda:0")

# 2. Get a candidate
uids = fn.get_diverse_objects(limit=1)
uid = uids[0]
print(f"   🎯 Testing Object UID: {uid}")

# 3. Download
try:
    objects = objaverse.load_objects(uids=[uid], download_processes=1)
    path = objects[uid]
    print("   ✅ Download: Success")
except:
    print("   ❌ Download: FAILED")

# 4. Convert (Trimesh)
temp_obj_path = f"debug_{uid}.obj"
try:
    mesh = trimesh.load(path, force='mesh')
    mesh.apply_translation(-mesh.centroid)
    scale = 1.0 / np.max(mesh.extents)
    mesh.apply_scale(scale)
    mesh.export(temp_obj_path)
    print("   ✅ Trimesh Conversion: Success")
except Exception as e:
    print(f"   ❌ Trimesh Conversion: FAILED ({e})")

# 5. Load to GPU
try:
    pytorch_mesh = load_objs_as_meshes([temp_obj_path], device=device)
    print("   ✅ PyTorch3D Load: Success")
except Exception as e:
    print(f"   ❌ PyTorch3D Load: FAILED ({e})")

# 6. Check Texture (Common silent failure point)
if pytorch_mesh.textures is None:
    print("   ⚠️ TEXTURE CHECK: FAILED (Mesh has no texture, would be skipped)")
else:
    print("   ✅ TEXTURE CHECK: Success (Mesh has texture)")

# 7. Initialize Renderer (Crucial Step)
try:
    dist = 2.5; elev = 30.0; azim = 0
    R, T = look_at_view_transform(dist=dist, elev=elev, azim=azim)
    cameras = FoVPerspectiveCameras(device=device, R=R, T=T)
    raster_settings = RasterizationSettings(image_size=512, blur_radius=0.0, faces_per_pixel=1)
    lights = PointLights(device=device, location=[[0.0, 0.0, -3.0]])

    renderer = MeshRenderer(
        rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
        shader=SoftPhongShader(device=device, cameras=cameras, lights=lights)
    )
    print("   ✅ Renderer Initialization: Success")
except Exception as e:
    print(f"   ❌ Renderer Initialization: FAILED ({e})")

# 8. PERFORM RENDER (The Moment of Truth)
print("   ⏳ Attempting to Rasterize (Draw) on L4 GPU...")
try:
    images = renderer(pytorch_mesh)
    print("   🎉🎉🎉 RENDER SUCCESS! The pipeline is working.")
except Exception as e:
    print("\n❌❌❌ RENDER FAILED!")
    print(f"Error Message: {e}")
    if "no kernel image is available" in str(e):
        print("\n💡 DIAGNOSIS: The PyTorch3D library is installed, but the CUDA Kernels are mismatching the L4 chip.")

🔍 Starting Deep Probe Diagnostic...
   🎯 Testing Object UID: 66b845b9bceb48ae9bc7ef401523dca4
Downloaded 1 / 1 objects
   ✅ Download: Success
   ✅ Trimesh Conversion: Success
   ✅ PyTorch3D Load: Success
   ✅ TEXTURE CHECK: Success (Mesh has texture)
   ✅ Renderer Initialization: Success
   ⏳ Attempting to Rasterize (Draw) on L4 GPU...
   🎉🎉🎉 RENDER SUCCESS! The pipeline is working.


In [ ]:
#@title Train Loop

# 1. Config
TOTAL_TARGET_IMAGES = 100
CHUNK_SIZE = 20
TRAIN_EPOCHS_PER_CHUNK = 5
BATCH_SIZE = 1
local_img_dir = "/content/training_data"
checkpoint_dir = f"{drive_path}/checkpoints"

# 2. Setup (If not already loaded)
if 'unet' not in globals():
    vae, tokenizer, text_encoder, unet, noise_scheduler = fn.load_stable_diffusion_model()
    optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-5)
    # Resume if possible
    start_steps, _ = fn.load_sd_checkpoint(unet, optimizer, checkpoint_dir)
else:
    # If resuming in same session
    print("Model already loaded in memory.")
    start_steps = 0 # Or track manually

images_processed_total = 0
unet.train()

print(f"🚀 Starting Training Loop. Goal: {TOTAL_TARGET_IMAGES} images.")

while images_processed_total < TOTAL_TARGET_IMAGES:

    # --- PHASE A: GENERATE ---
    print(f"\n📂 [Phase A] Generating Data Chunk... ({images_processed_total}/{TOTAL_TARGET_IMAGES})")
    if os.path.exists(local_img_dir): shutil.rmtree(local_img_dir)
    os.makedirs(local_img_dir, exist_ok=True)

    candidates = fn.get_diverse_objects(limit=CHUNK_SIZE * 3)
    current_chunk_images = []
    current_chunk_prompts = []

    for uid in candidates:
        if len(current_chunk_images) >= CHUNK_SIZE: break
        try:
            objects = objaverse.load_objects(uids=[uid], download_processes=1)
            path = objects[uid]
            if fn.render_views(path, local_img_dir, uid):
                img_path = os.path.join(local_img_dir, f"{uid}_combined.jpg")
                prompt = fn.generate_prompt_from_image(img_path)
                if "Error" not in prompt:
                    current_chunk_images.append(img_path)
                    current_chunk_prompts.append(prompt)
                    print(f"   + Added {uid}: '{prompt[:30]}...'")
                else:
                  print(f"Error: {prompt}.")
            if os.path.exists(path): os.remove(path)
        except: pass

    if not current_chunk_images: continue

    # --- PHASE B: TRAIN ---
    print(f"🏋️ [Phase B] Training on {len(current_chunk_images)} images...")

    # Use the Dataset Class from functions.py
    train_dataset = fn.MultiViewDataset(current_chunk_images, current_chunk_prompts, tokenizer)
    train_dataloader = fn.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    for epoch in range(TRAIN_EPOCHS_PER_CHUNK):
        epoch_loss = 0.0
        for batch in train_dataloader:
            # Use the Train Function from functions.py
            loss = fn.train_batch(batch, unet, vae, text_encoder, noise_scheduler, optimizer)
            epoch_loss += loss
        print(f"   Epoch {epoch+1} | Loss: {epoch_loss / len(train_dataloader):.4f}")

    # --- PHASE C: SAVE ---
    images_processed_total += len(current_chunk_images)
    fn.save_sd_checkpoint(unet, optimizer, images_processed_total, epoch_loss, checkpoint_dir)

    print(f"🧹 Chunk done. Total: {images_processed_total}")

Model already loaded in memory.
🚀 Starting Training Loop. Goal: 100 images.

📂 [Phase A] Generating Data Chunk... (0/100)
Downloaded 1 / 1 objects
   + Added 3603cf85c49e4323a93b62db0258b36f: 'A modern, low-profile bed feat...'
Downloaded 1 / 1 objects
   + Added 8c6ac61b545c4910a6d3f97cd59fc71c: 'A realistic depiction of a sle...'
Downloaded 1 / 1 objects
   + Added 8f8e1646cc1549c6ab22f1a3c0cd2924: 'A modern table lamp features a...'
Downloaded 1 / 1 objects
   + Added e7bf494054f545778647321fd9813bb8: 'A dark, inverted conical objec...'
Downloaded 1 / 1 objects
   + Added 8801b377a9934c1191b513561db845d5: 'A sturdy touring motorcycle fe...'
Downloaded 1 / 1 objects
   + Added 49d81cd52df6422a81132835c2555b2f: 'A robust, vintage-style tourin...'
Downloaded 1 / 1 objects
   + Added 99c784499f0b40c6943c91fafa707b0f: 'A black top hat with a tall, c...'
Downloaded 1 / 1 objects
   + Added 8bbefe57330941c6904f4e4618756553: 'A dark grey upholstered sofa w...'
Downloaded 1 / 1 objects
   + 

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


   + Added 728e03ecd80340f1ac62494b8d59bfb1: 'A sailboat featuring a dark gr...'
Downloaded 1 / 1 objects
   + Added 7d4cf08c9750498cabba8cb4b4dc2c7c: 'A vintage rangefinder camera f...'
🏋️ [Phase B] Training on 20 images...
   Epoch 1 | Loss: 0.0121
   Epoch 2 | Loss: 0.0092
   Epoch 3 | Loss: 0.0114
   Epoch 4 | Loss: 0.0076
   Epoch 5 | Loss: 0.0093
✅ SD Checkpoint saved: /content/drive/MyDrive/ Generative Models Course/checkpoints/sd_finetune.pth
🧹 Chunk done. Total: 100


In [ ]:
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt

def run_inference_test(prompt, unet, vae, tokenizer, text_encoder, noise_scheduler):
    """
    Generates an image using the current state of the trained UNet.
    """
    print(f"🎨 Generating test for: '{prompt}'...")

    # 1. Create a temporary pipeline with the TRAINED unet
    # We use safety_checker=None to save memory and avoid false positives
    pipe = StableDiffusionPipeline(
        vae=vae,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        unet=unet,
        scheduler=noise_scheduler,
        safety_checker=None,
        feature_extractor=None,
        requires_safety_checker=False
    ).to(fn.device)

    # 2. Generate
    # We use 30 steps for a quick but decent quality check
    image = pipe(prompt, num_inference_steps=30).images[0]

    # 3. Display
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Prompt: {prompt}")
    plt.show()

    # 4. Cleanup (to save VRAM for training)
    del pipe
    torch.cuda.empty_cache()

# --- RUN THE TEST ---
# Try a prompt relevant to your dataset
test_prompt = "sunglasses"
#test_prompt = "a blue airplane"

run_inference_test(test_prompt, unet, vae, tokenizer, text_encoder, noise_scheduler)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


NameError: name 'unet' is not defined

In [ ]:
#@title --- MAIN EXECUTION ---
# 1. Setup Folders
images_dir = f"{drive_path}/images"
os.makedirs(images_dir, exist_ok=True)

# 2. Get UIDs (Toys)
print("Fetching Toy UIDs...")
# We grab 5 toys to test. Change limit=50 later for real run.
uids = fn.get_toy_objects(limit=10)
print(f"Found {len(uids)} objects to process.")

# 3. Download Objects
print("Downloading objects (this may take a moment)...")
objects = objaverse.load_objects(uids=uids, download_processes=1)

# 4. Process Loop
print("Starting Rendering...")
for uid, path in objects.items():
    print(f"Processing: {uid}")
    fn.render_views(path, image_path, uid)

    try:
      os.remove(path)
      parent_dir = os.path.dirname(path)
      if not os.listdir(parent_dir):
        os.rmdir(parent_dir)

      print(f"  -> Cleaned up local 3D files to save space.")
    except Exception as e:
      print(f"  -> Error: Could not delete local file: {e}")


# fn.combine_images_grid(f"{drive_path}/images", image_path)
print("Done! Check your 'images' folder.")

Fetching Toy UIDs...
Found 10 objects to process.
Downloaded 1 / 10 objects
Downloaded 2 / 10 objects
Downloaded 3 / 10 objects
Downloaded 4 / 10 objects
Downloaded 5 / 10 objects
Downloaded 6 / 10 objects
Downloaded 7 / 10 objects
Downloaded 8 / 10 objects
Downloaded 9 / 10 objects
Downloaded 10 / 10 objects
Starting Rendering...
Processing: 7e684a7c012c4fd0ac91844f22457640
   -> Skipping 7e684a7c012c4fd0ac91844f22457640: Object has no texture.
  -> Cleaned up local 3D files to save space.
Processing: 4a40780198964bdeb6e9b6950a3c69fb
   Saved 4 views for 4a40780198964bdeb6e9b6950a3c69fb
  -> Cleaned up local 3D files to save space.
Processing: aec69c979166446eb2c8e1503f570d26
   Saved 4 views for aec69c979166446eb2c8e1503f570d26
  -> Cleaned up local 3D files to save space.
Processing: bdc8eb3e2a10409bafdfa85fca906204
   Saved 4 views for bdc8eb3e2a10409bafdfa85fca906204
  -> Cleaned up local 3D files to save space.
Processing: 862a2bfb08644ba79453bd86ed9874bb
   -> Skipping 862a2bfb0

In [ ]:
#@title generate prompts for all images in folder

# exponential backoff for quota limit
max_retries = 5
base_wait_time = 30

for filename in files:
  if filename.endswith(('.png', '.jpg', '.jpeg')):
    full_path = os.path.join(image_path, filename)
    print(f"Processing: {filename}...")

    success = False
    current_wait_time = base_wait_time

    for attempt in range(max_retries):
      prompt = fn.generate_toy_prompt_from_image(full_path)

      # check for limit error
      if "Quota exceeded" in prompt:
        print(f"  -> Hit rate limit. (Attempt {attempt+1}/{max_retries}). Cooling for {current_wait_time}s")
        print(f"    --> Error: {prompt}")
        time.sleep(current_wait_time)
        current_wait_time *= 2

      # check for other errors
      elif "Error" in prompt:
        print(f"  -> Unexpected Error: {prompt}")
        break

      # success!
      else:
        print(f"--> Prompt: {prompt}\n")
        success = True
        time.sleep(0.1)
        break

    if not success:
      print(f"  !!! Skipping {filename} - could not generate prompt after multiple attempts.\n")

Processing: c6c08606ed24421f82c465eda33bb6ee_combined.jpg...
--> Prompt: An olive green model biplane with RAF roundels.

Processing: 1ded9f27d0ae46e9a7f4c871eef7ed8e_combined.jpg...
--> Prompt: A pixelated red van and red and dark grey pickup truck.

Processing: 2a11fb8525ac40809d722d0547a00366_combined.jpg...
--> Prompt: Brown metal truck chassis with black wheels.

Processing: 4e88a88d9c17457fa75e09a404dc4e71_combined.jpg...


KeyboardInterrupt: 